# 📜 Peringkasan Otomatis Artikel Ormas Islam
## Menggunakan Algoritma IndoBERT

---

### 📋 Informasi Notebook

| Komponen | Detail |
|----------|--------|
| **Model** | `indobenchmark/indobert-base-p1` |
| **Pendekatan** | Extractive Text Summarization |
| **Dataset** | IndoSum + Ormas Liputan6 (~35K artikel) |
| **Evaluasi** | ROUGE-1, ROUGE-2, ROUGE-L |
| **Framework** | PyTorch + HuggingFace Transformers |

### 🗺️ Alur Notebook
1. ✅ Instalasi & Setup Lingkungan
2. ✅ Exploratory Data Analysis (EDA)
3. ✅ Preprocessing & Label Generation
4. ✅ Arsitektur Model IndoBERT
5. ✅ Training (2 Tahap)
6. ✅ Evaluasi ROUGE
7. ✅ Simpan & Muat Model
8. ✅ Demo Inferensi

> **💡 Tips VS Code:** Pilih kernel `Python (IndoBERT Skripsi)` di pojok kanan atas.
> Jalankan sel dengan **Shift+Enter** atau klik tombol ▶️.
> Untuk GPU: pastikan CUDA tersedia, atau gunakan Google Colab.

---
## 📦 LANGKAH 1: Instalasi Paket & Setup

Jalankan sel ini sekali untuk menginstal semua dependensi yang diperlukan.

In [ ]:
# Install semua paket yang diperlukan
# (Jalankan sekali, kemudian restart kernel jika diperlukan)
import subprocess, sys

packages = [
    'torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118',
    'transformers>=4.35.0',
    'datasets>=2.14.0',
    'rouge-score>=0.1.2',
    'accelerate>=0.24.0',
    'sentencepiece>=0.1.99',
    'Sastrawi>=1.0.1',
    'scikit-learn>=1.3.0',
    'plotly>=5.17.0',
    'tqdm>=4.66.0',
]

print('Menginstal paket...')
for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + pkg.split(),
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '❌'
    print(f'{status} {pkg.split()[0]}')

print('\n✅ Instalasi selesai!')

In [ ]:
# ── Import Utama ──────────────────────────────────────────────────
import os
import sys
import json
import re
import pickle
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score

from rouge_score import rouge_scorer as rs

warnings.filterwarnings('ignore')

# ── Konfigurasi Paths ─────────────────────────────────────────────
# Sesuaikan ROOT dengan lokasi folder proyek Anda
ROOT = Path('d:/Joki')  # Ubah sesuai path Anda
INDOSUM_DIR = ROOT / 'indosum'
ORMAS_CSV = ROOT / 'ormas_liputan6.csv'
LIPUTAN6_DIR = ROOT / 'liputan6_data' / 'canonical'
DATA_DIR = ROOT / 'data' / 'processed'
MODEL_DIR = ROOT / 'models' / 'indobert_summarizer'
LOG_DIR = ROOT / 'logs'

# Buat direktori jika belum ada
for d in [DATA_DIR, MODEL_DIR, LOG_DIR, ROOT / 'results']:
    d.mkdir(parents=True, exist_ok=True)

# Tambah src ke path
sys.path.insert(0, str(ROOT))

print('✅ Library berhasil di-import!')
print(f'   Versi PyTorch: {torch.__version__}')
print(f'   Versi Transformers: ', end='')
import transformers; print(transformers.__version__)

In [ ]:
# ── Cek Device (GPU/CPU) ──────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'🚀 GPU Tersedia: {gpu_name} ({gpu_mem:.1f} GB VRAM)')
    print(f'   CUDA Version: {torch.version.cuda}')
else:
    device = torch.device('cpu')
    print('💻 Menggunakan CPU')
    print('   ⚠️  Training akan lebih lambat. Pertimbangkan Google Colab untuk GPU gratis.')
    print('   Untuk mengaktifkan GPU (jika tersedia): install driver CUDA + pytorch GPU')

print(f'\n   Device aktif: {device}')

# Set seed untuk reprodusibilitas
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f'   Random seed: {SEED}')

---
## 🔍 LANGKAH 2: Exploratory Data Analysis (EDA)

Analisis mendalam terhadap dataset sebelum training.

In [ ]:
# ── EDA: Ormas Liputan6 CSV ───────────────────────────────────────
print('📊 Memuat dataset Ormas Liputan6...')
df = pd.read_csv(ORMAS_CSV)
print(f'\nShape   : {df.shape}')
print(f'Kolom   : {list(df.columns)}')
print(f'Missing : \n{df.isnull().sum()}')

df.head(3)

In [ ]:
# ── Statistik Panjang Teks ────────────────────────────────────────
df_clean = df.dropna(subset=['content', 'summary'])
df_clean['content_len'] = df_clean['content'].str.split().str.len()
df_clean['summary_len'] = df_clean['summary'].str.split().str.len()
df_clean['compression'] = 1 - df_clean['summary_len'] / df_clean['content_len']

print('📊 Statistik Dataset Ormas Liputan6:\n')
stats = df_clean[['content_len', 'summary_len', 'compression']].describe()
print(stats.round(2).to_string())

# Visualisasi distribusi
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Distribusi Dataset Ormas Liputan6', fontsize=14, fontweight='bold')

# Distribusi panjang artikel
axes[0].hist(df_clean['content_len'].clip(0, 800), bins=40, color='#3f51b5', alpha=0.8)
axes[0].set_title('Panjang Artikel (kata)')
axes[0].set_xlabel('Jumlah Kata')
axes[0].set_ylabel('Frekuensi')
axes[0].axvline(df_clean['content_len'].median(), color='red', linestyle='--',
                label=f'Median: {df_clean["content_len"].median():.0f}')
axes[0].legend()

# Distribusi panjang ringkasan
axes[1].hist(df_clean['summary_len'].clip(0, 150), bins=40, color='#e91e63', alpha=0.8)
axes[1].set_title('Panjang Ringkasan (kata)')
axes[1].set_xlabel('Jumlah Kata')
axes[1].axvline(df_clean['summary_len'].median(), color='red', linestyle='--',
                label=f'Median: {df_clean["summary_len"].median():.0f}')
axes[1].legend()

# Rasio kompresi
axes[2].hist(df_clean['compression'].clip(0, 1), bins=40, color='#4caf50', alpha=0.8)
axes[2].set_title('Rasio Kompresi')
axes[2].set_xlabel('Kompresi (1 = ringkas semua)')
axes[2].axvline(df_clean['compression'].median(), color='red', linestyle='--',
                label=f'Median: {df_clean["compression"].median():.2f}')
axes[2].legend()

plt.tight_layout()
plt.savefig(ROOT / 'results' / 'eda_ormas_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Grafik disimpan ke results/eda_ormas_distribution.png')

In [ ]:
# ── Kata Paling Sering Muncul ─────────────────────────────────────
from collections import Counter
import re

STOPWORDS_ID = set([
    'yang', 'dan', 'di', 'ke', 'dari', 'ini', 'itu', 'dengan', 'untuk',
    'adalah', 'dalam', 'tidak', 'akan', 'pada', 'juga', 'sebagai', 'atau',
    'oleh', 'dapat', 'saat', 'telah', 'sudah', 'karena', 'lebih', 'ada',
    'bahwa', 'kita', 'mereka', 'ia', 'nya', 'kami', 'kamu', 'saya', 'anda',
    'namun', 'tetapi', 'namun', 'jika', 'kalau', 'agar', 'supaya', 'hingga',
    'sampai', 'ketika', 'setelah', 'sebelum', 'menjadi', 'sangat', 'seperti'
])

# Ambil sample 1000 artikel
sample_content = df_clean['content'].dropna().sample(min(1000, len(df_clean)), random_state=42)
all_words = []
for text in sample_content:
    words = re.findall(r'\b[a-zA-Z]{3,}\b', text.lower())
    all_words.extend([w for w in words if w not in STOPWORDS_ID])

word_freq = Counter(all_words)
top_words = word_freq.most_common(20)

# Bar chart
words, freqs = zip(*top_words)
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(words[::-1], freqs[::-1], color=plt.cm.Blues(np.linspace(0.4, 0.9, 20)))
ax.set_title('20 Kata Paling Sering dalam Artikel Ormas Islam (tanpa stopwords)', fontsize=13)
ax.set_xlabel('Frekuensi')

# Tambah label nilai
for bar, freq in zip(bars, freqs[::-1]):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{freq:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(ROOT / 'results' / 'eda_top_words.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 kata Ormas Islam:')
for word, freq in top_words[:10]:
    print(f'  {word:20s}: {freq:,}')

---
## 🔧 LANGKAH 3: Preprocessing Data

### Strategi:
1. **IndoSum** → Label sudah tersedia (`gold_labels`)
2. **Ormas CSV** → Buat label ekstraktif menggunakan **Greedy Oracle**:
   Pilih kalimat yang paling meningkatkan ROUGE terhadap ringkasan abstraktif

### Greedy Oracle Algorithm:
```
FOR i = 1 to max_sentences:
    Pilih kalimat k yang belum dipilih yang PALING MENINGKATKAN ROUGE
    Jika tidak ada peningkatan, hentikan
```

In [ ]:
# ── Fungsi Preprocessing ──────────────────────────────────────────

def clean_text(text: str) -> str:
    """Bersihkan teks dari HTML, URL, dan whitespace berlebih."""
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'Advertisement', '', text)
    text = re.sub(r'BACA JUGA:.*?(?=\n|$)', '', text)
    text = re.sub(r'[^\w\s.,!?;:()\'"\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def split_sentences(text: str, min_len: int = 20) -> list:
    """Pecah teks menjadi daftar kalimat."""
    sents = re.split(r'(?<=[.!?])\s+', text)
    result = []
    for s in sents:
        for sub in s.split('\n'):
            sub = sub.strip()
            if len(sub) >= min_len:
                result.append(sub)
    return result


def greedy_oracle_labels(sentences, summary, max_select=3, scorer=None):
    """
    Buat label ekstraktif menggunakan Greedy Oracle.
    Referensi: Liu & Lapata (2019)
    """
    if scorer is None:
        scorer = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    
    labels = [0] * len(sentences)
    selected = []
    
    for _ in range(min(max_select, len(sentences))):
        best_score = -1
        best_idx = -1
        
        for i, sent in enumerate(sentences):
            if labels[i] == 1:
                continue
            
            candidate = ' '.join(sentences[j] for j in sorted(selected + [i]))
            scores = scorer.score(summary, candidate)
            combined = (
                scores['rouge1'].fmeasure + 
                scores['rouge2'].fmeasure + 
                scores['rougeL'].fmeasure
            ) / 3
            
            if combined > best_score:
                best_score = combined
                best_idx = i
        
        if best_idx >= 0:
            labels[best_idx] = 1
            selected.append(best_idx)
    
    return labels


# Test fungsi
sample_article = """Nahdlatul Ulama (NU) merupakan organisasi Islam terbesar di Indonesia.
NU didirikan pada 31 Januari 1926 di Surabaya oleh para ulama pesantren.
Organisasi ini memiliki lebih dari 90 juta anggota yang tersebar di seluruh Indonesia.
NU berperan penting dalam menjaga keutuhan NKRI dan moderasi Islam."""

sample_summary = "NU adalah organisasi Islam terbesar Indonesia, didirikan 1926."

sents = split_sentences(clean_text(sample_article))
labels = greedy_oracle_labels(sents, sample_summary)

print('✅ Test Preprocessing')
print(f'   Jumlah kalimat: {len(sents)}')
print(f'   Label: {labels}')
print('\nKalimat terpilih:')
for i, (s, l) in enumerate(zip(sents, labels)):
    icon = '✅' if l == 1 else '  '
    print(f'  {icon} [{i}] {s[:80]}...')

In [ ]:
# ── Muat Dataset IndoSum ──────────────────────────────────────────
# IndoSum sudah memiliki label ekstraktif (gold_labels)

def load_indosum(indosum_dir, split='train'):
    """Muat dataset IndoSum dari JSONL."""
    import json
    examples = []
    
    for part in range(1, 6):
        filepath = Path(indosum_dir) / f'{split}.0{part}.jsonl'
        if not filepath.exists():
            continue
        
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    data = json.loads(line)
                except:
                    continue
                
                sentences, labels = [], []
                for para_idx, paragraph in enumerate(data.get('paragraphs', [])):
                    para_labels = data.get('gold_labels', [[]])
                    pl = para_labels[para_idx] if para_idx < len(para_labels) else []
                    
                    for sent_idx, tokens in enumerate(paragraph):
                        text = ' '.join(tokens).strip()
                        if len(text) < 10:
                            continue
                        sentences.append(text)
                        if sent_idx < len(pl):
                            labels.append(1 if pl[sent_idx] else 0)
                        else:
                            labels.append(0)
                
                if len(sentences) >= 2 and sum(labels) > 0:
                    examples.append({
                        'id': data.get('id', ''),
                        'sentences': sentences[:50],
                        'labels': labels[:50],
                        'category': data.get('category', ''),
                        'source': 'indosum'
                    })
    return examples


print('Memuat IndoSum train...')
indosum_train = load_indosum(INDOSUM_DIR, 'train')
print(f'   IndoSum train: {len(indosum_train)} dokumen')

print('Memuat IndoSum dev...')
indosum_dev = load_indosum(INDOSUM_DIR, 'dev')
print(f'   IndoSum dev: {len(indosum_dev)} dokumen')

print('Memuat IndoSum test...')
indosum_test = load_indosum(INDOSUM_DIR, 'test')
print(f'   IndoSum test: {len(indosum_test)} dokumen')

# Preview
ex = indosum_train[0]
print(f'\nContoh IndoSum (id={ex["id"]}):')
print(f'  Jumlah kalimat: {len(ex["sentences"])}')
print(f'  Label: {ex["labels"][:10]}...')
print(f'  Kalimat positif: {sum(ex["labels"])}')

In [ ]:
# ── Buat Label Ekstraktif untuk Ormas CSV ─────────────────────────
# Proses ini bisa memakan waktu 10-30 menit tergantung jumlah data

ORMAS_CACHE = DATA_DIR / 'ormas_processed_5000.pkl'
MAX_ORMAS_SAMPLES = 5000  # Gunakan 5000 untuk training (tambah jika ada lebih banyak waktu)

if ORMAS_CACHE.exists():
    print(f'📂 Memuat dari cache: {ORMAS_CACHE}')
    with open(ORMAS_CACHE, 'rb') as f:
        ormas_data = pickle.load(f)
    print(f'   Dimuat: {len(ormas_data)} contoh')
else:
    print(f'⚙️  Membuat label ekstraktif untuk {MAX_ORMAS_SAMPLES} artikel ormas...')
    print('   (Estimasi waktu: ~15-30 menit dengan CPU)\n')
    
    df_ormas = pd.read_csv(ORMAS_CSV)
    df_ormas = df_ormas.dropna(subset=['content', 'summary'])
    df_ormas = df_ormas[df_ormas['content'].str.len() > 100]
    df_ormas = df_ormas.sample(min(MAX_ORMAS_SAMPLES, len(df_ormas)), random_state=42)
    df_ormas = df_ormas.reset_index(drop=True)
    
    scorer = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    ormas_data = []
    skipped = 0
    
    for idx, row in tqdm(df_ormas.iterrows(), total=len(df_ormas), desc='Preprocessing'):
        content = clean_text(str(row['content']))
        summary = clean_text(str(row['summary']))
        
        sents = split_sentences(content)
        if len(sents) < 2:
            skipped += 1
            continue
        
        labels = greedy_oracle_labels(sents, summary, scorer=scorer)
        
        if sum(labels) == 0:
            skipped += 1
            continue
        
        ormas_data.append({
            'id': str(row.get('url', idx)),
            'title': str(row.get('title', '')),
            'summary': summary,
            'sentences': sents[:50],
            'labels': labels[:50],
            'source': 'ormas'
        })
    
    # Simpan cache
    with open(ORMAS_CACHE, 'wb') as f:
        pickle.dump(ormas_data, f)
    
    print(f'\n✅ Selesai! {len(ormas_data)} contoh diproses, {skipped} dilewati')
    print(f'   Cache disimpan ke: {ORMAS_CACHE}')

# Visualisasi distribusi label
all_labels_flat = [l for ex in ormas_data for l in ex['labels']]
pos_ratio = sum(all_labels_flat) / len(all_labels_flat)
print(f'\n📊 Rasio label positif (Ormas): {pos_ratio:.3f} ({pos_ratio*100:.1f}%)')
print(f'   Positif (kalimat penting): {sum(all_labels_flat):,}')
print(f'   Negatif (kalimat biasa): {len(all_labels_flat) - sum(all_labels_flat):,}')

In [ ]:
# ── Gabungkan & Bagi Dataset ──────────────────────────────────────
print('Menggabungkan dataset...')
all_data = indosum_train + indosum_dev + ormas_data
np.random.shuffle(all_data)
print(f'Total gabungan: {len(all_data)} dokumen')
print(f'  - IndoSum: {len(indosum_train) + len(indosum_dev)}')
print(f'  - Ormas: {len(ormas_data)}')

# Split 80/10/10
np.random.seed(SEED)
n = len(all_data)
train_end = int(n * 0.80)
val_end = int(n * 0.90)

train_data = all_data[:train_end]
val_data = all_data[train_end:val_end]
test_data = all_data[val_end:]

print(f'\nSplit Dataset:')
print(f'  Train : {len(train_data):,} ({len(train_data)/n:.0%})')
print(f'  Val   : {len(val_data):,} ({len(val_data)/n:.0%})')
print(f'  Test  : {len(test_data):,} ({len(test_data)/n:.0%})')

# Simpan split
for name, data in [('train', train_data), ('val', val_data), ('test', test_data)]:
    path = DATA_DIR / f'combined_{name}.pkl'
    with open(path, 'wb') as f:
        pickle.dump(data, f)
    print(f'  Disimpan: {path}')

print('\n✅ Data siap untuk training!')

---
## 🤖 LANGKAH 4: Arsitektur Model IndoBERT

### Arsitektur: BertSum-Style Extractive Summarizer

```
Input Dokumen
     ↓
[Kalimat 1] [Kalimat 2] ... [Kalimat N]
     ↓           ↓               ↓
  IndoBERT   IndoBERT         IndoBERT
  (shared)   (shared)         (shared)
     ↓           ↓               ↓
  Embedding1  Embedding2  ...  EmbeddingN   (768-dim)
     ↓────────────────────────────↓
        Positional Encoding
            ↓
     Inter-Sentence Transformer (2L)
            ↓
     Linear Classifier
            ↓
     Skor1  Skor2  ...  SkorN   [0, 1]
            ↓
     Top-K Selection → Ringkasan
```

Referensi: Liu & Lapata (2019) *"Text Summarization with Pretrained Encoders"*

In [ ]:
# ── Definisi Model ────────────────────────────────────────────────
import math

class PositionalEncoding(nn.Module):
    """Sinusoidal Positional Encoding untuk urutan kalimat."""
    def __init__(self, d_model, dropout=0.1, max_len=512):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).float().unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class IndoBERTSumExtractor(nn.Module):
    """
    IndoBERT Extractive Summarizer
    
    Pipeline:
        Kalimat → IndoBERT → Mean Pooling → Pos. Encoding
            → Inter-Sentence Transformer → Linear → Sigmoid
    """
    
    BERT_MODEL = 'indobenchmark/indobert-base-p1'
    
    def __init__(self, num_transformer_layers=2, num_heads=8, 
                 dropout=0.1, freeze_layers=8):
        super().__init__()
        
        # Load IndoBERT
        self.bert = AutoModel.from_pretrained(self.BERT_MODEL)
        self.hidden_size = self.bert.config.hidden_size  # 768
        
        # Bekukan N layer awal
        if freeze_layers > 0:
            for param in self.bert.embeddings.parameters():
                param.requires_grad = False
            for layer in self.bert.encoder.layer[:freeze_layers]:
                for param in layer.parameters():
                    param.requires_grad = False
        
        # Positional encoding
        self.pos_enc = PositionalEncoding(self.hidden_size, dropout)
        
        # Inter-sentence transformer
        enc_layer = nn.TransformerEncoderLayer(
            d_model=self.hidden_size,
            nhead=num_heads,
            dim_feedforward=2048,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(
            enc_layer, num_layers=num_transformer_layers,
            enable_nested_tensor=False
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(self.hidden_size, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )
        self.dropout_layer = nn.Dropout(dropout)
    
    def mean_pool(self, token_embs, attn_mask):
        mask = attn_mask.unsqueeze(-1).float()
        return (token_embs * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    
    def forward(self, input_ids, attention_mask, sent_mask=None):
        """
        Args:
            input_ids: (B, S, T) — Batch, kalimat, token
            attention_mask: (B, S, T)
            sent_mask: (B, S) — True = padding kalimat
        Returns:
            scores: (B, S) — Skor relevansi [0,1]
        """
        B, S, T = input_ids.shape
        
        # Enkode semua kalimat sekaligus
        out = self.bert(input_ids.view(-1, T), attention_mask.view(-1, T))
        sent_embs = self.mean_pool(out.last_hidden_state, attention_mask.view(-1, T))
        sent_embs = sent_embs.view(B, S, -1)  # (B, S, H)
        sent_embs = self.pos_enc(self.dropout_layer(sent_embs))
        
        # Inter-sentence attention
        sent_embs = self.transformer(sent_embs, src_key_padding_mask=sent_mask)
        
        # Klasifikasi
        scores = torch.sigmoid(self.classifier(sent_embs).squeeze(-1))
        return scores
    
    def count_params(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total, trainable


# Inisialisasi & tampilkan info
print('⏳ Memuat model IndoBERT (pertama kali akan download ~500MB)...')
model = IndoBERTSumExtractor(freeze_layers=8).to(device)
total_params, trainable_params = model.count_params()

print(f'\n✅ Model berhasil dimuat!')
print(f'   Total parameter   : {total_params:,}')
print(f'   Parameter dilatih : {trainable_params:,}')
print(f'   Proporsi dilatih  : {trainable_params/total_params:.1%}')

# Test forward pass
dummy_ids = torch.randint(0, 1000, (2, 5, 32)).to(device)
dummy_mask = torch.ones(2, 5, 32, dtype=torch.long).to(device)
dummy_sent_mask = torch.zeros(2, 5, dtype=torch.bool).to(device)

with torch.no_grad():
    out = model(dummy_ids, dummy_mask, dummy_sent_mask)

print(f'   Test output shape: {out.shape}')  # Expected: (2, 5)
print(f'   Skor contoh: {out[0].cpu().tolist()}')

---
## 🏋️ LANGKAH 5: Training Model

### Strategi 2 Tahap:
- **Tahap 1**: BERT dibekukan → Latih head cepat (5 epoch, LR tinggi)
- **Tahap 2**: BERT dicairkan → Fine-tune end-to-end (3 epoch, LR kecil)

### Loss: Weighted BCE
Karena kalimat positif (penting) << negatif, kita tambahkan bobot 3× untuk kelas positif.

$$\mathcal{L} = -\frac{1}{N}\sum_i \left[ w_{pos} \cdot y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i) \right]$$

In [ ]:
# ── Dataset & DataLoader ──────────────────────────────────────────
MAX_SENT_LEN = 128
MAX_SENTENCES = 40
BATCH_SIZE = 4

print(f'⏳ Memuat tokenizer IndoBERT...')
tokenizer = AutoTokenizer.from_pretrained('indobenchmark/indobert-base-p1')
print(f'✅ Tokenizer dimuat (vocab size: {tokenizer.vocab_size:,})')


class SumDataset(Dataset):
    def __init__(self, examples, tokenizer, max_sent_len=128, max_sents=40):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_sent_len = max_sent_len
        self.max_sents = max_sents
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        ex = self.examples[idx]
        sents = ex['sentences'][:self.max_sents]
        labels = ex['labels'][:self.max_sents]
        
        enc = self.tokenizer(
            sents, padding='max_length', truncation=True,
            max_length=self.max_sent_len, return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'],
            'attention_mask': enc['attention_mask'],
            'labels': torch.tensor(labels, dtype=torch.float32),
            'n_sents': len(sents)
        }


def collate_fn(batch):
    max_sents = max(b['n_sents'] for b in batch)
    max_len = batch[0]['input_ids'].shape[1]
    
    ids_list, mask_list, lbl_list, smask_list = [], [], [], []
    for b in batch:
        n = b['n_sents']
        pad = max_sents - n
        
        ids = b['input_ids']
        msk = b['attention_mask']
        lbl = b['labels']
        
        if pad > 0:
            z = torch.zeros(pad, max_len, dtype=torch.long)
            ids = torch.cat([ids, z])
            msk = torch.cat([msk, z])
            lbl = torch.cat([lbl, torch.zeros(pad)])
        
        smask = torch.zeros(max_sents, dtype=torch.bool)
        smask[n:] = True
        
        ids_list.append(ids)
        mask_list.append(msk)
        lbl_list.append(lbl)
        smask_list.append(smask)
    
    return {
        'input_ids': torch.stack(ids_list),
        'attention_mask': torch.stack(mask_list),
        'labels': torch.stack(lbl_list),
        'sent_mask': torch.stack(smask_list),
        'n_sents': [b['n_sents'] for b in batch]
    }


train_ds = SumDataset(train_data, tokenizer, MAX_SENT_LEN, MAX_SENTENCES)
val_ds   = SumDataset(val_data,   tokenizer, MAX_SENT_LEN, MAX_SENTENCES)
test_ds  = SumDataset(test_data,  tokenizer, MAX_SENT_LEN, MAX_SENTENCES)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'\n📊 DataLoader siap:')
print(f'   Train: {len(train_loader)} batch ({len(train_data)} dokumen)')
print(f'   Val  : {len(val_loader)} batch ({len(val_data)} dokumen)')
print(f'   Test : {len(test_loader)} batch ({len(test_data)} dokumen)')

In [ ]:
# ── Fungsi Training & Evaluasi ────────────────────────────────────
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

class WeightedBCE(nn.Module):
    def __init__(self, pos_weight=3.0):
        super().__init__()
        self.pos_weight = pos_weight
    
    def forward(self, pred, target):
        eps = 1e-7
        pred = pred.clamp(eps, 1 - eps)
        return (-(self.pos_weight * target * torch.log(pred) + 
                  (1 - target) * torch.log(1 - pred))).mean()


def run_epoch(model, loader, criterion, optimizer=None, scheduler=None, 
              device=device, grad_accum=4, desc='Train'):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    
    total_loss, all_preds, all_labels = 0, [], []
    if is_train:
        optimizer.zero_grad()
    
    with torch.set_grad_enabled(is_train):
        for step, batch in enumerate(tqdm(loader, desc=desc, leave=False)):
            ids = batch['input_ids'].to(device)
            msk = batch['attention_mask'].to(device)
            lbl = batch['labels'].to(device)
            smask = batch['sent_mask'].to(device)
            
            scores = model(ids, msk, smask)
            valid = ~smask
            loss = criterion(scores[valid], lbl[valid])
            
            if is_train:
                (loss / grad_accum).backward()
                if (step + 1) % grad_accum == 0:
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    if scheduler: scheduler.step()
                    optimizer.zero_grad()
            
            total_loss += loss.item()
            preds = (scores[valid].detach() > 0.5).float().cpu().numpy()
            lbls = lbl[valid].detach().cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(lbls.tolist())
    
    y_t = np.array(all_labels)
    y_p = np.array(all_preds)
    return {
        'loss': total_loss / max(len(loader), 1),
        'f1': float(f1_score(y_t, y_p, zero_division=0)),
        'precision': float(precision_score(y_t, y_p, zero_division=0)),
        'recall': float(recall_score(y_t, y_p, zero_division=0)),
        'accuracy': float(accuracy_score(y_t, y_p)),
    }

print('✅ Fungsi training & evaluasi didefinisikan')

In [ ]:
# ════════════════════════════════════════════════════════════════
#  TAHAP 1: Training Head (BERT Terbekukan)
# ════════════════════════════════════════════════════════════════
# Konfigurasi
NUM_EPOCHS_1 = 5
LR_1 = 5e-4
GRAD_ACCUM = 4
POS_WEIGHT = 3.0

criterion = WeightedBCE(POS_WEIGHT)
opt1 = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR_1, weight_decay=0.01
)
total_steps = len(train_loader) * NUM_EPOCHS_1 // GRAD_ACCUM
sched1 = get_linear_schedule_with_warmup(
    opt1, int(total_steps * 0.1), total_steps
)

history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}
best_val_f1 = 0.0

print('🏋️ TAHAP 1: Training Head...')
print(f'   Epoch: {NUM_EPOCHS_1} | LR: {LR_1} | Batch: {BATCH_SIZE}×{GRAD_ACCUM}')
print('─' * 65)

for epoch in range(1, NUM_EPOCHS_1 + 1):
    t0 = time.time()
    
    tr = run_epoch(model, train_loader, criterion, opt1, sched1, 
                   device=device, grad_accum=GRAD_ACCUM, desc=f'Epoch {epoch} Train')
    vl = run_epoch(model, val_loader, criterion, 
                   device=device, desc=f'Epoch {epoch} Val')
    
    elapsed = time.time() - t0
    history['train_loss'].append(tr['loss'])
    history['val_loss'].append(vl['loss'])
    history['train_f1'].append(tr['f1'])
    history['val_f1'].append(vl['f1'])
    
    print(f'  Ep {epoch:2d}/{NUM_EPOCHS_1} | {elapsed:.0f}s | '
          f'Loss: {tr["loss"]:.4f}/{vl["loss"]:.4f} | '
          f'F1: {tr["f1"]:.4f}/{vl["f1"]:.4f} | '
          f'P: {vl["precision"]:.4f} R: {vl["recall"]:.4f}')
    
    if vl['f1'] > best_val_f1:
        best_val_f1 = vl['f1']
        torch.save(model.state_dict(), MODEL_DIR / 'model_best.pt')
        print(f'     ✅ Best model disimpan (F1: {best_val_f1:.4f})')
    
    # Simpan checkpoint per epoch
    torch.save(model.state_dict(), MODEL_DIR / f'model_epoch{epoch}.pt')

print(f'\n✅ Tahap 1 selesai! Best Val F1: {best_val_f1:.4f}')

In [ ]:
# ════════════════════════════════════════════════════════════════
#  TAHAP 2: Fine-tuning Penuh (BERT Tidak Terbekukan)
# ════════════════════════════════════════════════════════════════
NUM_EPOCHS_2 = 3
LR_2 = 2e-5
BERT_LR = 1e-5

# Cairkan semua parameter BERT
for param in model.bert.parameters():
    param.requires_grad = True

total_params, trainable_params = model.count_params()
print(f'Parameter dilatih sekarang: {trainable_params:,} / {total_params:,}')

# Optimizer dengan LR berbeda untuk BERT vs head
bert_params = list(model.bert.parameters())
head_params = list(model.transformer.parameters()) + list(model.classifier.parameters())

opt2 = torch.optim.AdamW([
    {'params': bert_params, 'lr': BERT_LR},
    {'params': head_params, 'lr': LR_2},
], weight_decay=0.01)

total_steps2 = len(train_loader) * NUM_EPOCHS_2 // GRAD_ACCUM
sched2 = get_cosine_schedule_with_warmup(
    opt2, int(total_steps2 * 0.1), total_steps2
)

print('\n🏋️ TAHAP 2: Fine-tuning Penuh...')
print(f'   Epoch: {NUM_EPOCHS_2} | BERT LR: {BERT_LR} | Head LR: {LR_2}')
print('─' * 65)

for epoch in range(1, NUM_EPOCHS_2 + 1):
    t0 = time.time()
    ep_real = epoch + NUM_EPOCHS_1
    
    tr = run_epoch(model, train_loader, criterion, opt2, sched2,
                   device=device, grad_accum=GRAD_ACCUM, desc=f'Epoch {ep_real} Train')
    vl = run_epoch(model, val_loader, criterion,
                   device=device, desc=f'Epoch {ep_real} Val')
    
    elapsed = time.time() - t0
    history['train_loss'].append(tr['loss'])
    history['val_loss'].append(vl['loss'])
    history['train_f1'].append(tr['f1'])
    history['val_f1'].append(vl['f1'])
    
    print(f'  Ep {ep_real:2d}/{NUM_EPOCHS_1+NUM_EPOCHS_2} | {elapsed:.0f}s | '
          f'Loss: {tr["loss"]:.4f}/{vl["loss"]:.4f} | '
          f'F1: {tr["f1"]:.4f}/{vl["f1"]:.4f}')
    
    if vl['f1'] > best_val_f1:
        best_val_f1 = vl['f1']
        torch.save(model.state_dict(), MODEL_DIR / 'model_best.pt')
        print(f'     ✅ Best model diperbarui (F1: {best_val_f1:.4f})')

# Simpan model final dan tokenizer
torch.save(model.state_dict(), MODEL_DIR / 'model_final.pt')
tokenizer.save_pretrained(str(MODEL_DIR))

config_data = {'bert_model_name': 'indobenchmark/indobert-base-p1', 'hidden_size': 768}
with open(MODEL_DIR / 'config.json', 'w') as f:
    json.dump(config_data, f, indent=2)

print(f'\n✅ Training lengkap!')
print(f'   Best Val F1: {best_val_f1:.4f}')
print(f'   Model tersimpan di: {MODEL_DIR}')

---
## 📊 LANGKAH 6: Visualisasi Training & Evaluasi ROUGE

In [ ]:
# ── Plot Kurva Training ───────────────────────────────────────────
epochs = list(range(1, len(history['train_loss']) + 1))
stage1_end = NUM_EPOCHS_1  # Batas tahap 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Kurva Training IndoBERT Summarizer', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train Loss', markersize=4)
axes[0].plot(epochs, history['val_loss'], 'r-o', label='Val Loss', markersize=4)
axes[0].axvline(x=stage1_end + 0.5, color='gray', linestyle='--', alpha=0.7)
axes[0].text(stage1_end + 0.6, max(history['train_loss']) * 0.95,
             'Stage 1 → Stage 2', fontsize=9, color='gray')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# F1
axes[1].plot(epochs, history['train_f1'], 'b-o', label='Train F1', markersize=4)
axes[1].plot(epochs, history['val_f1'], 'r-o', label='Val F1', markersize=4)
axes[1].axvline(x=stage1_end + 0.5, color='gray', linestyle='--', alpha=0.7)
axes[1].set_title('F1 Score')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / 'results' / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Kurva training disimpan ke results/training_curves.png')

In [ ]:
# ── Evaluasi ROUGE pada Test Set ──────────────────────────────────
# Muat best model
model.load_state_dict(torch.load(MODEL_DIR / 'model_best.pt', map_location=device))
model.eval()
print('✅ Best model dimuat')

scorer = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

hypotheses, references = [], []

print('Menghitung ROUGE pada test set...')
for example in tqdm(test_data[:500], desc='Evaluasi ROUGE'):  # Ambil 500 sample
    sents = example['sentences'][:MAX_SENTENCES]
    labels = example['labels'][:MAX_SENTENCES]
    if not sents:
        continue
    
    enc = tokenizer(sents, padding='max_length', truncation=True,
                    max_length=MAX_SENT_LEN, return_tensors='pt')
    ids = enc['input_ids'].unsqueeze(0).to(device)
    msk = enc['attention_mask'].unsqueeze(0).to(device)
    
    with torch.no_grad():
        scores = model(ids, msk)[0].cpu().numpy()
    
    # Pilih top-30% kalimat
    n = len(sents)
    k = max(1, min(5, int(n * 0.3)))
    top_k = sorted(range(n), key=lambda i: scores[i], reverse=True)[:k]
    hypothesis = ' '.join(sents[i] for i in sorted(top_k))
    
    # Referensi dari gold labels
    gold_idxs = [i for i, l in enumerate(labels) if l == 1]
    if gold_idxs:
        reference = ' '.join(sents[i] for i in sorted(gold_idxs))
    elif 'summary' in example:
        reference = example['summary']
    else:
        continue
    
    hypotheses.append(hypothesis)
    references.append(reference)

# Hitung rata-rata ROUGE
rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
for h, r in zip(hypotheses, references):
    s = scorer.score(r, h)
    rouge_scores['rouge1'].append(s['rouge1'].fmeasure)
    rouge_scores['rouge2'].append(s['rouge2'].fmeasure)
    rouge_scores['rougeL'].append(s['rougeL'].fmeasure)

r1 = np.mean(rouge_scores['rouge1'])
r2 = np.mean(rouge_scores['rouge2'])
rl = np.mean(rouge_scores['rougeL'])

print('\n' + '=' * 50)
print('  📊 HASIL EVALUASI ROUGE (Test Set)')
print('=' * 50)
print(f'  ROUGE-1 F1  : {r1:.4f} ({r1*100:.2f}%)')
print(f'  ROUGE-2 F1  : {r2:.4f} ({r2*100:.2f}%)')
print(f'  ROUGE-L F1  : {rl:.4f} ({rl*100:.2f}%)')
print('=' * 50)

# Simpan hasil
results = {
    'rouge1_f': r1, 'rouge2_f': r2, 'rougeL_f': rl,
    'num_samples': len(hypotheses)
}
with open(ROOT / 'results' / 'evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'  Hasil disimpan ke results/evaluation_results.json')

In [ ]:
# ── Visualisasi ROUGE ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hasil Evaluasi ROUGE', fontsize=14, fontweight='bold')

# Bar chart ROUGE scores
metrics = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
scores_avg = [r1, r2, rl]
colors = ['#3f51b5', '#e91e63', '#4caf50']

bars = axes[0].bar(metrics, scores_avg, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
for bar, score in zip(bars, scores_avg):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{score:.4f}', ha='center', va='bottom', fontweight='bold')
axes[0].set_ylim(0, max(scores_avg) * 1.25)
axes[0].set_title('Skor ROUGE Rata-rata (Test Set)')
axes[0].set_ylabel('F1 Score')
axes[0].grid(axis='y', alpha=0.3)

# Distribusi ROUGE-1 F1
axes[1].hist(rouge_scores['rouge1'], bins=30, color='#3f51b5', alpha=0.7, label='ROUGE-1')
axes[1].hist(rouge_scores['rouge2'], bins=30, color='#e91e63', alpha=0.5, label='ROUGE-2')
axes[1].hist(rouge_scores['rougeL'], bins=30, color='#4caf50', alpha=0.4, label='ROUGE-L')
axes[1].axvline(r1, color='#1a237e', linestyle='--', linewidth=1.5)
axes[1].axvline(r2, color='#880e4f', linestyle='--', linewidth=1.5)
axes[1].axvline(rl, color='#1b5e20', linestyle='--', linewidth=1.5)
axes[1].set_title('Distribusi ROUGE Scores')
axes[1].set_xlabel('F1 Score')
axes[1].set_ylabel('Frekuensi')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / 'results' / 'rouge_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Evaluasi ROUGE selesai!')

---
## 🚀 LANGKAH 7: Demo Inferensi

Uji model dengan artikel baru.

In [ ]:
# ── Demo Inferensi ────────────────────────────────────────────────
def summarize(text, model, tokenizer, ratio=0.3, min_sents=1, max_sents=5,
              device=device, max_sent_len=128, max_sentences=40):
    """Hasilkan ringkasan dari teks artikel."""
    # Preprocessing
    clean = clean_text(text)
    sentences = split_sentences(clean)
    if not sentences:
        return text, [], []
    
    sents = sentences[:max_sentences]
    
    # Tokenisasi
    enc = tokenizer(sents, padding='max_length', truncation=True,
                    max_length=max_sent_len, return_tensors='pt')
    ids = enc['input_ids'].unsqueeze(0).to(device)
    msk = enc['attention_mask'].unsqueeze(0).to(device)
    
    # Prediksi
    model.eval()
    with torch.no_grad():
        scores = model(ids, msk)[0].cpu().numpy()
    
    # Pilih kalimat
    n = len(sents)
    k = max(min_sents, min(max_sents, int(n * ratio)))
    top_k = sorted(range(n), key=lambda i: scores[i], reverse=True)[:k]
    selected = sorted(top_k)
    summary = ' '.join(sents[i] for i in selected)
    
    return summary, sents, scores.tolist(), selected


# ── Artikel Contoh ────────────────────────────────────────────────
test_article = """
Liputan6.com, Jakarta - Nahdlatul Ulama (NU) kembali menegaskan komitmennya 
sebagai organisasi Islam terbesar di Indonesia untuk selalu berkontribusi dalam 
pembangunan bangsa. Ketua Umum PBNU mengungkapkan hal tersebut dalam acara 
Harlah ke-99 NU yang digelar di Jakarta.

Dalam sambutannya, Ketua PBNU menekankan pentingnya peran ulama dan santri dalam 
memperkuat persatuan nasional di tengah berbagai tantangan global. NU selama hampir 
satu abad telah membuktikan diri sebagai garda terdepan dalam menjaga keutuhan NKRI 
dan Pancasila.

Sementara itu, Muhammadiyah juga menyatakan dukungannya terhadap program moderasi 
beragama yang dijalankan pemerintah. Ketua PP Muhammadiyah menegaskan bahwa Islam 
rahmatan lil alamin harus diwujudkan dalam kehidupan berbangsa dan bernegara.

Kedua organisasi Islam terbesar di Indonesia ini sepakat bahwa toleransi dan 
kerukunan antarumat beragama merupakan fondasi utama bagi kemajuan Indonesia. 
Mereka berkomitmen untuk terus bekerja sama dalam mencegah radikalisme dan 
ekstremisme yang mengancam persatuan bangsa.

Menteri Agama turut mengapresiasi kontribusi NU dan Muhammadiyah dalam memperkuat 
moderasi beragama di Indonesia. Pemerintah berharap kedua ormas ini terus menjadi 
pilar utama dalam mewujudkan Indonesia yang damai, adil, dan makmur.
"""

print('📄 ARTIKEL ASLI:')
print('─' * 60)
print(test_article.strip())

print('\n📋 RINGKASAN (IndoBERT, 30%):')
print('─' * 60)
summary, sents, scores, selected = summarize(test_article, model, tokenizer, ratio=0.3)
print(summary)

print(f'\n📊 STATISTIK:')
orig_words = len(test_article.split())
summ_words = len(summary.split())
print(f'  Kalimat asli   : {len(sents)}')
print(f'  Kalimat dipilih: {len(selected)} (indeks: {selected})')
print(f'  Kata asli      : {orig_words}')
print(f'  Kata ringkasan : {summ_words}')
print(f'  Kompresi       : {(1 - summ_words/orig_words)*100:.1f}%')

In [ ]:
# ── Visualisasi Skor Kalimat ──────────────────────────────────────
if sents and scores:
    selected_set = set(selected)
    colors = ['#2e7d32' if i in selected_set else '#90a4ae' for i in range(len(sents))]
    
    fig, ax = plt.subplots(figsize=(12, 5))
    x = range(len(sents))
    bars = ax.bar(x, scores[:len(sents)], color=colors, alpha=0.85, edgecolor='white')
    
    ax.set_title('Skor Relevansi Kalimat - IndoBERT Summarizer', fontsize=13, fontweight='bold')
    ax.set_xlabel('Indeks Kalimat')
    ax.set_ylabel('Skor Relevansi (0-1)')
    ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Threshold 0.5')
    ax.set_ylim(0, 1.2)
    ax.grid(axis='y', alpha=0.3)
    
    # Tambah label nilai
    for bar, score in zip(bars, scores[:len(sents)]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{score:.2f}', ha='center', va='bottom', fontsize=8)
    
    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#2e7d32', label='Kalimat Terpilih'),
        Patch(facecolor='#90a4ae', label='Kalimat Tidak Dipilih'),
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    plt.savefig(ROOT / 'results' / 'inference_scores.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Tampilkan kalimat dengan skor
    print('\nDaftar kalimat dan skor:')
    for i, (sent, score) in enumerate(zip(sents, scores[:len(sents)])):
        icon = '✅' if i in selected_set else '  '
        print(f'{icon} [{i}] ({score:.3f}) {sent[:80]}...')

---
## 🎯 Ringkasan & Kesimpulan

### Apa yang Telah Kita Lakukan:

| Langkah | Status | Keterangan |
|---------|--------|------------|
| 1. Setup & Instalasi | ✅ | PyTorch, Transformers, ROUGE |
| 2. EDA | ✅ | Analisis 35K artikel Ormas Islam |
| 3. Preprocessing | ✅ | Greedy Oracle label generation |
| 4. Model IndoBERT | ✅ | BertSum + Inter-Sentence Transformer |
| 5. Training (2 Tahap) | ✅ | Frozen BERT → Full fine-tune |
| 6. Evaluasi ROUGE | ✅ | ROUGE-1, ROUGE-2, ROUGE-L |
| 7. Simpan Model | ✅ | Checkpoint + Tokenizer |
| 8. Demo Inferensi | ✅ | Artikel Ormas Islam baru |

### Cara Menjalankan Streamlit App:
```bash
# Aktifkan environment
venv\Scripts\activate

# Jalankan aplikasi
streamlit run app/streamlit_app.py
```

### Referensi:
- Liu, Y. & Lapata, M. (2019). *Text Summarization with Pretrained Encoders*. EMNLP 2019.
- Wilie et al. (2020). *IndoNLU: Benchmark and Resources for Evaluating Indonesian NLP*.
- Kurniawan, K. & Louvan, S. (2018). *IndoSum: A New Benchmark for Indonesian Automatic Text Summarization*.